# Week 4 Assignment: Machine Learning for Forecasting
**DSE6230: Forecasting Methods and Applications**

This assignment focuses on using Machine Learning (XGBoost) and Facebook Prophet for forecasting. You will use `mlforecast` and `statsforecast` from Nixtla.

1. **Feature Engineering**: Lags, Rolling Windows, Date Features.
2. **XGBoost**: Training a Gradient Boosted Tree.
3. **Prophet**: Training an Additive Model.
4. **Evaluation**: Compare ML vs Prophet.

In [ ]:
!pip install pandas mlforecast statsforecast xgboost datasetsforecast prophet window-ops

In [18]:
import pandas as pd
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences, LocalStandardScaler
from statsforecast import StatsForecast
# note: statsforecast does not have a native Prophet import. see more in the wrapper below
from prophet import Prophet as FBProphet
from xgboost import XGBRegressor
from datasetsforecast.m4 import M4
# imported window ops
from window_ops.rolling import rolling_mean
# imported tools
import logging
import warnings
import time
import numpy as np
import matplotlib.pyplot as plt

In [16]:
# WRAPPER FOR PROPHET
# A wrapper for making Prophet work without using the AutoARIMA version provided by statsforecast. Made w/ assistance from AI for proper formatting.
warnings.filterwarnings("ignore")
for name in ("cmdstanpy", "prophet"):
    logging.getLogger(name).setLevel(logging.ERROR)

class Prophet:
    uses_exog = False

    def __init__(self, freq="D", alias="Prophet", **prophet_kwargs):
        self.freq = freq
        self.alias = alias
        self.prophet_kwargs = prophet_kwargs

    def __repr__(self):
        return self.alias

    def new(self):
        b = type(self).__new__(type(self))
        b.__dict__.update(self.__dict__)
        return b

    def forecast(self, y, h, X=None, X_future=None, level=None, fitted=False):
        for name in ("cmdstanpy", "prophet"):          # silence logs inside worker processes too
            logging.getLogger(name).setLevel(logging.ERROR)
        ds = pd.date_range("2000-01-01", periods=len(y), freq=self.freq)
        m = FBProphet(**self.prophet_kwargs).fit(pd.DataFrame({"ds": ds, "y": y}))
        future = m.make_future_dataframe(periods=h, freq=self.freq, include_history=False)
        return {"mean": m.predict(future)["yhat"].to_numpy()}


## Part 1: Self-Guided Exercises

### Exercise 1.1: Feature Engineering
**Instruction**: Initialize an `MLForecast` object with:
- `models`: `[XGBRegressor()]`
- `freq`: 'D'
- `lags`: [1, 7, 14]
- `lag_transforms`: {1: [window_ops.mean]} (Use Rolling Mean of window 7) *Note: Import `window_ops` first if needed, or skip transforms for basic check.*

In [20]:
# Your Code Here
# fcst = MLForecast(...)
fcst = MLForecast(
    models=[XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4)],
    freq="D",
    lags=[1, 7, 14],
    lag_transforms={1: [(rolling_mean, 7)]},
)

### Exercise 1.2: Prophet
**Instruction**: Use `StatsForecast` to fit a `Prophet` model to a sample dataset (create a dummy df if needed).

In [21]:
# Your Code Here
# sf = StatsForecast(models=[Prophet()], freq='D')

# Dummy dataframe
rng = np.random.default_rng(42)
dates = pd.date_range("2023-01-01", periods=365, freq="D")
dummy_df = pd.concat([
    pd.DataFrame({
        "unique_id": uid,
        "ds": dates,
        "y": level + 0.05 * np.arange(365) + 10 * np.sin(2 * np.pi * dates.dayofweek / 7)
             + rng.normal(0, 2, 365),
    })
    for uid, level in [("series_A", 100), ("series_B", 50)]
], ignore_index=True)
dummy_df.head()

# Preprocessing
features = fcst.preprocess(dummy_df)
print(features.columns.tolist())
features.head()

# Create the test / train split
h = 14
dummy_test = dummy_df.groupby("unique_id").tail(h)
dummy_train = dummy_df.drop(dummy_test.index)

# Prophet Code
sf = StatsForecast(models=[Prophet(freq="D")], freq="D")
prophet_dummy = sf.forecast(df=dummy_train, h=h)

res = dummy_test.merge(prophet_dummy, on=["unique_id", "ds"])
print(f"Prophet MAE on dummy data: {(res.y - res.Prophet).abs().mean():.3f}")
prophet_dummy.head()


['unique_id', 'ds', 'y', 'lag1', 'lag7', 'lag14', 'rolling_mean_lag1_window_size7']
Prophet MAE on dummy data: 1.643


,unique_id,ds,Prophet
0,series_A,2023-12-18,117.897832
1,series_A,2023-12-19,125.688990
2,series_A,2023-12-20,127.840469
3,series_A,2023-12-21,121.421930
4,series_A,2023-12-22,113.481536


## Part 2: Case Study
Download the **M4 Hourly** dataset (or a subset).

1. Train an XGBoost model using `mlforecast`.
2. Train a Prophet model using `statsforecast`.
3. Compare their MAE on a 48-hour horizon.
4. **Analysis**: Which model was faster? Which was more accurate? Why?

In [22]:
# Download the M4 Hourly dataset
group_name = 'Hourly'
Y_df, *_ = M4.load(directory='data', group=group_name)
# Your Code Here

# Convert from integers to time
Y_df["ds"] = pd.Timestamp("2020-01-01") + pd.to_timedelta(Y_df["ds"].astype(int) - 1, unit="h")

# Hold out the last 48 hours for testing
H = 48
test_df = Y_df.groupby("unique_id").tail(H)
train_df = Y_df.drop(test_df.index)
print(f"{Y_df.unique_id.nunique()} series | train rows: {len(train_df):,} | test rows: {len(test_df):,}")

2.35MiB [00:00, 42.4MiB/s]                 
133kiB [00:00, 28.5MiB/s]                   
4.34MiB [00:00, 81.2MiB/s]                 
100%|██████████| 3.56M/3.56M [00:00<00:00, 39.2MiB/s]


414 series | train rows: 353,500 | test rows: 19,872


In [23]:
# XGBOOST MODEL
mlf = MLForecast(
    models=[XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, n_jobs=-1)],
    freq="h",
    lags=[1, 2, 24, 48, 168],
    lag_transforms={
        1: [(rolling_mean, 24)], 24: [(rolling_mean, 168)],
    },
    date_features=["hour", "dayofweek"],
    target_transforms=[LocalStandardScaler()],
)

# Time the training and prediction
start = time.time()
mlf.fit(train_df)
xgb_fc = mlf.predict(H)
xgb_time = time.time() - start
print(f"XGBoost train + predict time: {xgb_time:.1f} s")
xgb_fc.head()

XGBoost train + predict time: 6.9 s


,unique_id,ds,XGBRegressor
0,H1,2020-01-30 04:00:00,653.750094
1,H1,2020-01-30 05:00:00,600.083463
2,H1,2020-01-30 06:00:00,553.646606
3,H1,2020-01-30 07:00:00,524.574341
4,H1,2020-01-30 08:00:00,504.963155


In [26]:
# PROPHET MODEL
sf = StatsForecast(
    models=[Prophet(freq="h", daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=False)], # -> wrapper class
    freq="h",
    n_jobs=1,
)

# Time the training
start = time.time()
prophet_fc = sf.forecast(df=train_df, h=H) # -> forecast is from the wrapper's function
prophet_time = time.time() - start
print(f"Prophet train + predict time: {prophet_time:.1f} s")
prophet_fc.head()

Prophet train + predict time: 112.7 s


,unique_id,ds,Prophet
0,H1,2020-01-30 04:00:00,668.028584
1,H1,2020-01-30 05:00:00,608.600940
2,H1,2020-01-30 06:00:00,556.796280
3,H1,2020-01-30 07:00:00,521.134712
4,H1,2020-01-30 08:00:00,502.292094


In [27]:
# RESULTS

# Merge together the two data frames
results = test_df.merge(xgb_fc, on=["unique_id", "ds"]).merge(prophet_fc, on=["unique_id", "ds"])
assert len(results) == len(test_df)

# Define MAE and sMAPE for display
def mae(y, yhat):
    return np.mean(np.abs(y - yhat))

def smape(y, yhat):
    return 100 * np.mean(2 * np.abs(y - yhat) / (np.abs(y) + np.abs(yhat)))

# The data frame to display
summary = pd.DataFrame({
    "MAE": [mae(results.y, results.XGBRegressor), mae(results.y, results.Prophet)],
    "sMAPE (%)": [smape(results.y, results.XGBRegressor), smape(results.y, results.Prophet)],
    "Time (s)": [xgb_time, prophet_time],
}, index=["XGBoost (mlforecast)", "Prophet (statsforecast)"]).round(2)

# Per-series comparison: which model has the lower MAE on each series?
per_series = results.groupby("unique_id").apply(
    lambda d: pd.Series({"XGBoost": mae(d.y, d.XGBRegressor), "Prophet": mae(d.y, d.Prophet)})
)
xgb_wins = (per_series.XGBoost < per_series.Prophet).sum()
print(f"XGBoost has the lower MAE on {xgb_wins} of {len(per_series)} series")
summary

# ANALYSIS
# Between our models, XGBoost was significantly faster than Prophet. XGBoost took merely seven seconds to predict, while Prophet went on for nearly two minutes.
# In terms of accuracy, XGBoost also wins - with a MAE of 358.38 against Prophet's 387.60, we see notably more success, just as with the sMAPE (13% vs 18).
# Though XGBoost only has the lower MAE on 189/414 series (or roughly 45%) overall, due to the irregular data set, it's quite likely that XGBoost handles better
# on average, whereas Prophet has a significantly higher ceiling for failure. This leads to an overall better MAE, and marking XGBoost as our better option overall.

XGBoost has the lower MAE on 189 of 414 series


,MAE,sMAPE (%),Time (s)
XGBoost (mlforecast),358.38,13.03,6.92
Prophet (statsforecast),387.60,18.10,112.72


## Grading Rubric
- **Correctness (40%)**: Does the code run?
- **Code Quality (30%)**: Is the feature engineering step clear?
- **Analysis (30%)**: Comparison of XGBoost vs Prophet.